# 101 Spark basics

The goal of this lab is to get familiar with Spark programming.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [14]:
from pyspark.sql import SparkSession

In [15]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

<SparkContext master=local appName=Local Spark>

## 101-1 Spark warm-up

Load the ```capra``` and ```divinacommedia``` datasets and try the following actions:
- Show their content (```collect```)
- Count their rows (```count```)
- Split phrases into words (```map``` or ```flatMap```; what’s the difference?)
- Check the results (remember: evaluation is lazy)
- Try the ```toDebugString``` function to check the execution plan
    - In PySpark, use ```toDebugString().decode("unicode_escape")```

In [16]:
rddCapra = sc.textFile("work/datasets/capra.txt")
rddDC = sc.textFile("work/datasets/divinacommedia.txt")

In [17]:
rddDC.collect()
rddCapra.collect()

['sopra la panca la capra campa', 'sotto la panca la capra crepa']

In [18]:
rddCapra.count()

2

In [19]:
rddDC.count()

14753

In [20]:
capraMapped = rddCapra.map(lambda w:w.split(" "))
capraFlatMapped = rddCapra.flatMap(lambda w: w.split(" "))
capraMapped.collect()
capraFlatMapped.collect()

['sopra',
 'la',
 'panca',
 'la',
 'capra',
 'campa',
 'sotto',
 'la',
 'panca',
 'la',
 'capra',
 'crepa']

In [21]:
print(rddCapra.map(lambda w: w.split(" ")).toDebugString().decode("unicode_escape"))

(1) PythonRDD[103] at RDD at PythonRDD.scala:53 []
 |  work/datasets/capra.txt MapPartitionsRDD[96] at textFile at NativeMethodAccessorImpl.java:0 []
 |  work/datasets/capra.txt HadoopRDD[95] at textFile at NativeMethodAccessorImpl.java:0 []


## 101-2 Basic Spark jobs

Implement on Spark the following jobs and test them on both capra and divinacommedia datasets.

- **Word count**: count the number of occurrences of each word
  - Result: (sopra, 1), (la, 4), …
- **Word length count**: count the number of occurrences of words of given lengths
  - Result: (2, 4), (5, 8)
- Count the average length of words given their first letter (i.e., words that begin with "s" have an average length of 5)
  - Result: (s, 5), (l, 2), …
- Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...

Also, check how sorting works and try to sort key-value RDDs by descending values.

In [ ]:
# word count
rddCapra.flatMap(lambda w: w.split(" ")).map(lambda w: (w, 1)).reduceByKey(lambda x, y: x + y).collect()
rddDC.flatMap(lambda w: w.split(" ")).map(lambda w: (w, 1)).reduceByKey(lambda x, y: x + y).collect()

#word length count
rddCapra.flatMap(lambda w: w.split(" ")).map(lambda w: (len(w), 1)).reduceByKey(lambda x, y: x + y).collect()
rddDC.flatMap(lambda w: w.split(" ")).map(lambda w: (len(w), 1)).reduceByKey(lambda x, y: x + y).collect()

#average length count
rddCapra \
    .flatMap(lambda w: w.split(" ")) \
    .map(lambda w: (w[0], (len(w), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .map(lambda x: (x[0], x[1][0]/x[1][1])) \
    .collect()
rddDC \
    .flatMap(lambda w: w.split(" ")) \
    .filter(lambda w: len(w) > 0) \
    .map(lambda w: (w[0], (len(w), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .map(lambda x: (x[0], x[1][0]/x[1][1])).collect()  #oppure mapValues(lambda v: v[0]/v[1]), lavoro solo sui valori

#list of lines number for each word
seqFunc = lambda acc, v: acc | {v}
combFunc = lambda acc1, acc2: acc1 | acc2
rddCapra \
    .zipWithIndex() \
    .flatMap(lambda w: [(word, w[1]) for word in w[0].split(" ")]) \
    .aggregateByKey(set(), seqFunc, combFunc) \
    .collect()
rddDC \
    .zipWithIndex() \
    .flatMap(lambda w: [(word, w[1]) for word in w[0].split(" ")]) \
    .aggregateByKey(set(), seqFunc, combFunc) \
    .collect()

#oppure switcho keys and values e uso flatMapvalues, e poi li riswitcho
rddCapra \
    .zipWithIndex() \
    .flatMap(lambda w: [(word, w[1]) for word in w[0].split(" ")]) \
    .groupByKey() \
    .mapValues(lambda v: set(v)) \
    .collect()

#sorting by descending values
rddCapra \
    .flatMap(lambda w: w.split(" ")) \
    .zipWithIndex().map(lambda x: (x[1], x[0])) \
    .sortByKey(False).map(lambda x: (x[1], x[0])) \
    .collect()


[('sopra', {0}),
 ('la', {0, 1}),
 ('panca', {0, 1}),
 ('capra', {0, 1}),
 ('campa', {0}),
 ('sotto', {1}),
 ('crepa', {1})]

## 101-3 Extra Spark jobs

Implement the following job.

- Co-occurrence count: count the number of co-occurrences in the text. A co-occurrence is defined as "two distinct words appearing in the same line".
  - In the first line of the *capra* dataset, co-occurrences are:
     - (sopra, la), (sopra, panca), (sopra, capra), (sopra, campa)
     - (la, sopra), (la, panca), (la, capra), (la, campa) 
     - (panca, sopra), (panca, la), (panca, capra), (panca, campa)
     - (capra, sopra), (capra, la), (capra, panca), (capra, campa)
     - (campa, sopra), (campa, la), (campa, panca), (campa, capra)

In [24]:
coOccurrences = rddDC.map(lambda row: len(list(set(row.split(" "))))*(len(list(set(row.split(" ")))) - 1)).sum()
print(coOccurrences)

623840
